# ObjC Design Patterns

This notebook teaches four foundational patterns used in server-side Objective-C code,
particularly when building an ATProto Personal Data Server (PDS).

**What you'll learn:**
- **Service objects** encapsulate business rules; controller code stays thin
- **Repositories** encapsulate data access; services delegate to them
- **Dispatch tables** route method names to handler functions
- **Validation guards** check inputs at system boundaries

These patterns are not specific to any one project—they're widely used in service-oriented ObjC code.

**Estimated Time:** 15-20 minutes

## Pattern 1: Service Objects

Services encapsulate business logic. They validate inputs, enforce rules, and delegate data access to repositories. A record service validates records, ensures required fields are present, and delegates storage to a repository.

In [ ]:
// Service object pattern — the service owns business rules
@interface RecordService : NSObject
- (BOOL)createRecord:(NSDictionary *)record forDid:(NSString *)did;
- (NSDictionary *)getRecord:(NSString *)collection forDid:(NSString *)did;
@end

@implementation RecordService
- (BOOL)createRecord:(NSDictionary *)record forDid:(NSString *)did {
    // Business rule: records must have a collection
    NSString *collection = [record objectForKey:@"collection"];
    if (collection == nil || [collection length] == 0) {
        NSLog(@"Rejected: record has no collection");
        return NO;
    }
    NSLog(@"Created record in %@ for %@", collection, did);
    return YES;
}
- (NSDictionary *)getRecord:(NSString *)collection forDid:(NSString *)did {
    NSLog(@"Fetching %@ for %@", collection, did);
    return @{@"collection": collection, @"did": did};
}
@end

RecordService *svc = [[RecordService alloc] init];
[svc createRecord:@{@"collection": @"app.bsky.feed.post", @"text": @"Hello"} forDid:@"did:plc:abc"];
[svc createRecord:@{@"text": @"No collection"} forDid:@"did:plc:abc"];

## Pattern 2: Repository Delegation

Repositories encapsulate data access. Services call repositories for all storage and retrieval operations. This separation means services can be tested without a real database, and data access logic stays centralized.

In [ ]:
// Repository pattern — owns data storage, services delegate to it
@interface RecordRepository : NSObject
@property (nonatomic, strong) NSMutableDictionary *store;
- (BOOL)storeRecord:(NSDictionary *)record atKey:(NSString *)key;
- (NSDictionary *)fetchRecordAtKey:(NSString *)key;
@end

@implementation RecordRepository
- (instancetype)init {
    self = [super init];
    if (self) {
        _store = [NSMutableDictionary dictionary];
    }
    return self;
}
- (BOOL)storeRecord:(NSDictionary *)record atKey:(NSString *)key {
    if (key == nil || record == nil) return NO;
    [_store setObject:record forKey:key];
    return YES;
}
- (NSDictionary *)fetchRecordAtKey:(NSString *)key {
    return [_store objectForKey:key];
}
@end

// Service uses repository for storage
RecordRepository *repo = [[RecordRepository alloc] init];
[repo storeRecord:@{@"text": @"Hello"} atKey:@"app.bsky.feed.post/abc"];
NSDictionary *fetched = [repo fetchRecordAtKey:@"app.bsky.feed.post/abc"];
NSLog(@"Fetched: %@", fetched);

## Pattern 3: Dispatch Tables

A dispatcher maps method names to handler functions. When an RPC request arrives, the dispatcher looks up the handler by name and executes it. This pattern scales: adding a new method means registering a new handler, not modifying dispatch logic.

In [ ]:
// Dispatch table pattern — route method names to handler blocks
@interface MethodDispatcher : NSObject
@property (nonatomic, strong) NSMutableDictionary *handlers;
- (void)registerMethod:(NSString *)methodId handler:(id)handler;
- (NSDictionary *)dispatchMethod:(NSString *)methodId params:(NSDictionary *)params;
@end

@implementation MethodDispatcher
- (instancetype)init {
    self = [super init];
    if (self) {
        _handlers = [NSMutableDictionary dictionary];
    }
    return self;
}
- (void)registerMethod:(NSString *)methodId handler:(id)handler {
    [_handlers setObject:handler forKey:methodId];
    NSLog(@"Registered: %@", methodId);
}
- (NSDictionary *)dispatchMethod:(NSString *)methodId params:(NSDictionary *)params {
    id handler = [_handlers objectForKey:methodId];
    if (handler == nil) {
        NSLog(@"No handler for %@", methodId);
        return @{@"error": @"MethodNotFound"};
    }
    // In real code, handler is a block. Here we log the dispatch.
    NSLog(@"Dispatched %@ with params: %@", methodId, params);
    return @{@"status": @"ok", @"method": methodId};
}
@end

MethodDispatcher *disp = [[MethodDispatcher alloc] init];
[disp registerMethod:@"com.atproto.server.createSession" handler:@"sessionHandler"];
[disp registerMethod:@"com.atproto.repo.createRecord" handler:@"recordHandler"];

NSDictionary *result = [disp dispatchMethod:@"com.atproto.server.createSession" params:@{@"handle": @"alice.bsky.social"}];
NSLog(@"Result: %@", result);

// Unknown method returns error
NSDictionary *missing = [disp dispatchMethod:@"com.atproto.unknown" params:@{}];
NSLog(@"Missing: %@", missing);

## Pattern 4: Validation Guards

Validate inputs at the system boundary, before they reach service logic. This guards against invalid handles, malformed DIDs, and out-of-range values. Keep validation logic separate from business logic.

In [ ]:
// Validation guard pattern — check inputs at the boundary
@interface HandleValidator : NSObject
- (BOOL)isValidHandle:(NSString *)handle;
@end

@implementation HandleValidator
- (BOOL)isValidHandle:(NSString *)handle {
    if (handle == nil || [handle length] < 3) return NO;
    if ([handle length] > 253) return NO;
    // Must contain at least one dot
    if (![handle containsString:@"."]) return NO;
    // No spaces allowed
    if ([handle containsString:@" "]) return NO;
    return YES;
}
@end

HandleValidator *validator = [[HandleValidator alloc] init];
NSLog(@"alice.bsky.social: %d", [validator isValidHandle:@"alice.bsky.social"]);
NSLog(@"invalid: %d", [validator isValidHandle:@"invalid"]);
NSLog(@"has space: %d", [validator isValidHandle:@"has space.com"]);
NSLog(@"too short: %d", [validator isValidHandle:@"ab"]);

## Exercise

Add a `listRecords` method to `RecordService` that returns all records for a given collection.
The service should delegate to the repository's `store` dictionary.

Hint: iterate over the repository's store keys, filter by collection prefix,
and collect matching records into an NSMutableArray.

In [ ]:
// Exercise: implement listRecords:forDid:
// Your code here...

## Solution

In [ ]:
// Solution: listRecords filters repository keys by collection prefix
@interface RecordService2 : NSObject
@property (nonatomic, strong) RecordRepository *repository;
- (NSMutableArray *)listRecords:(NSString *)collection forDid:(NSString *)did;
@end

@implementation RecordService2
- (NSMutableArray *)listRecords:(NSString *)collection forDid:(NSString *)did {
    NSMutableArray *results = [NSMutableArray array];
    NSString *prefix = [NSString stringWithFormat:@"%@/", collection];
    for (NSString *key in _repository.store) {
        if ([key hasPrefix:prefix]) {
            [results addObject:[_repository fetchRecordAtKey:key]];
        }
    }
    return results;
}
@end

// Test it
RecordService2 *svc2 = [[RecordService2 alloc] init];
svc2.repository = [[RecordRepository alloc] init];
[svc2.repository storeRecord:@{@"text": @"Hello"} atKey:@"app.bsky.feed.post/abc"];
[svc2.repository storeRecord:@{@"text": @"World"} atKey:@"app.bsky.feed.post/def"];
[svc2.repository storeRecord:@{@"name": @"Alice"} atKey:@"app.bsky.actor.profile/xyz"];

NSMutableArray *posts = [svc2 listRecords:@"app.bsky.feed.post" forDid:@"did:plc:abc"];
NSLog(@"Posts: %d", [posts count]);

## Next Steps

These patterns combine to form the backbone of an ATProto PDS. You'll see them again as you build:
- Account services, session managers, and blob storage
- XRPC endpoint dispatch and input validation
- Repository diffing and repository synchronization

Move on to the ATProto notebooks to see these patterns applied to real protocol concepts.